# Grad-CAM XAI Evaluation

Reproduces the XAI results from:

> **Interpretable Deep Learning for Plant Disease Detection: A Comparative Study with Explainability Metrics**  
> Joy et al., ICoEIT 2025, IEEE. DOI: 10.1109/ICoEIT.2025.01

This notebook:
1. Loads the four trained CNN models
2. Generates Grad-CAM heatmaps for each model
3. Computes Jaccard Index against ground-truth annotation masks (Table VI)
4. Computes the novel **Explainability Score** (Equation 6) per disease class
5. Generates the annotation sampling list for LabelMe

**Prerequisites:** Run `train.py` first to produce the `.keras` model files,  
or download pretrained weights from the release page.

## 0. Configuration — set your paths here

In [ ]:
import os

# ── Edit these two lines only ─────────────────────────────────────────────────
DATA_DIR   = "path/to/newplantdisease_subset"   # root with train/ and valid/
MODELS_DIR = "path/to/saved_models"             # folder containing .keras files
# ─────────────────────────────────────────────────────────────────────────────

VALID_DIR  = os.path.join(DATA_DIR, "valid")
MASKS_DIR  = os.path.join(DATA_DIR, "organized_masks")  # LabelMe output folder

CLASS_NAMES = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
]

# Last conv layer name per model (required for Grad-CAM)
LAST_CONV_LAYERS = {
    "InceptionV3":  "mixed10",
    "DenseNet121":  "conv5_block16_concat",
    "Xception":     "block14_sepconv2_act",
    "MobileNetV2":  "out_relu",
}

print("Config loaded.")
print(f"  Data dir  : {DATA_DIR}")
print(f"  Models dir: {MODELS_DIR}")

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing import image as keras_image
import os
import random
import csv
from pathlib import Path

print(f"TensorFlow version: {tf.__version__}")

## 2. Load trained models

In [ ]:
models = {
    "InceptionV3":  tf.keras.models.load_model(os.path.join(MODELS_DIR, "inceptionv3_final.keras")),
    "DenseNet121":  tf.keras.models.load_model(os.path.join(MODELS_DIR, "densenet121_final.keras")),
    "Xception":     tf.keras.models.load_model(os.path.join(MODELS_DIR, "xception_final.keras")),
    "MobileNetV2":  tf.keras.models.load_model(os.path.join(MODELS_DIR, "mobilenetv2_final.keras")),
}

for name, model in models.items():
    print(f"{name}: {model.count_params():,} parameters")

## 3. Grad-CAM implementation

In [ ]:
def load_image(img_path: str) -> np.ndarray:
    """Load and preprocess a single image to (1, 224, 224, 3)."""
    img = keras_image.load_img(img_path, target_size=(224, 224))
    arr = keras_image.img_to_array(img) / 255.0
    return np.expand_dims(arr, axis=0)


def grad_cam(model, img_array: np.ndarray, last_conv_layer_name: str) -> np.ndarray:
    """
    Generate a Grad-CAM heatmap.

    Args:
        model: Trained Keras model.
        img_array: Preprocessed image, shape (1, 224, 224, 3).
        last_conv_layer_name: Name of final convolutional layer.

    Returns:
        heatmap: Normalised ndarray, shape (224, 224), values in [0, 1].
    """
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output],
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_class = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_class]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap).numpy()
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    return cv2.resize(heatmap, (224, 224))


def overlay_heatmap(img_path: str, heatmap: np.ndarray, alpha: float = 0.3) -> np.ndarray:
    """
    Overlay Grad-CAM heatmap on original image.
    Uses COLORMAP_INFERNO (colorblind-friendly) instead of JET.
    """
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_INFERNO)
    return cv2.addWeighted(img, 1 - alpha, heatmap_color, alpha, 0)


print("Grad-CAM functions defined.")

## 4. XAI metrics — Jaccard Index and Explainability Score

In [ ]:
def jaccard_index(mask_a: np.ndarray, mask_b: np.ndarray, threshold: float = 0.5) -> float:
    """
    Jaccard Index (IoU) between two masks — Equation 5 of the paper.
    |A ∩ B| / |A ∪ B|
    """
    a = (mask_a >= threshold).astype(bool)
    b = (mask_b >= threshold).astype(bool)
    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(intersection / union) if union > 0 else 0.0


def explainability_score(heatmaps: list, threshold: float = 0.5) -> float:
    """
    Explainability Score — Equation 6 of the paper.

    Average pairwise Jaccard similarity between Grad-CAM heatmaps
    of images from the same disease class. Higher = more consistent
    disease-relevant attention.

    ES = (2 / n(n-1)) * Σ_i Σ_{j>i}  |H_i ∩ H_j| / |H_i ∪ H_j|

    Args:
        heatmaps: List of heatmaps (np.ndarray H×W), same disease class.
        threshold: Binarisation threshold.

    Returns:
        Explainability Score in [0, 1].
    """
    n = len(heatmaps)
    if n < 2:
        raise ValueError("Need at least 2 heatmaps to compute Explainability Score.")
    total = sum(
        jaccard_index(heatmaps[i], heatmaps[j], threshold)
        for i in range(n - 1)
        for j in range(i + 1, n)
    )
    return total / (n * (n - 1) / 2)


print("Metric functions defined.")

## 5. Visualise Grad-CAM heatmaps for one sample image per model

Reproduces Figure 4 of the paper (correctly classified samples).

In [ ]:
# Pick any image from the validation set
SAMPLE_CLASS = "Tomato___Late_blight"
sample_dir   = os.path.join(VALID_DIR, SAMPLE_CLASS)
sample_img   = os.path.join(sample_dir, os.listdir(sample_dir)[0])

img_array = load_image(sample_img)
original  = cv2.cvtColor(cv2.imread(sample_img), cv2.COLOR_BGR2RGB)
original  = cv2.resize(original, (224, 224))

fig, axes = plt.subplots(1, len(models) + 1, figsize=(20, 4))
axes[0].imshow(original)
axes[0].set_title("Original", fontsize=11)
axes[0].axis("off")

for ax, (model_name, model) in zip(axes[1:], models.items()):
    heatmap  = grad_cam(model, img_array, LAST_CONV_LAYERS[model_name])
    overlaid = overlay_heatmap(sample_img, heatmap)
    overlaid = cv2.cvtColor(overlaid, cv2.COLOR_BGR2RGB)
    ax.imshow(overlaid)
    ax.set_title(model_name, fontsize=11)
    ax.axis("off")

fig.suptitle(f"Grad-CAM heatmaps — {SAMPLE_CLASS}", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("gradcam_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: gradcam_comparison.png")

## 6. Jaccard Index — heatmap vs ground-truth mask

Reproduces Table VI (left column) of the paper.  
Requires LabelMe annotation masks in `MASKS_DIR`.

In [ ]:
def compute_jaccard_vs_mask(
    model,
    model_name: str,
    class_name: str,
    n_samples: int = 10,
    threshold: float = 0.5,
) -> float:
    """
    Average Jaccard Index between Grad-CAM heatmaps and
    expert-annotated ground truth masks for a disease class.
    """
    mask_class_dir = os.path.join(MASKS_DIR, class_name)
    img_class_dir  = os.path.join(VALID_DIR, class_name)

    if not os.path.exists(mask_class_dir):
        print(f"  No masks found for {class_name} — skipping.")
        return None

    mask_files = [f for f in os.listdir(mask_class_dir) if f.endswith("_mask.png")]
    mask_files = mask_files[:n_samples]

    jaccard_scores = []
    for mask_file in mask_files:
        img_name  = mask_file.replace("_mask.png", ".JPG")
        img_path  = os.path.join(img_class_dir, img_name)
        mask_path = os.path.join(mask_class_dir, mask_file)

        if not os.path.exists(img_path):
            continue

        img_array = load_image(img_path)
        heatmap   = grad_cam(model, img_array, LAST_CONV_LAYERS[model_name])

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = cv2.resize(gt_mask, (224, 224)) / 255.0

        jaccard_scores.append(jaccard_index(heatmap, gt_mask, threshold))

    return float(np.mean(jaccard_scores)) if jaccard_scores else None


print("Jaccard evaluation function defined.")
print("NOTE: Run section 8 (annotation pipeline) first if masks don't exist yet.")

In [ ]:
# Compute Jaccard Index for all models × all classes
# Skip this cell if annotation masks are not available

jaccard_results = {}
for model_name, model in models.items():
    print(f"\n── {model_name} ─────────────────────")
    scores = []
    for class_name in CLASS_NAMES:
        score = compute_jaccard_vs_mask(model, model_name, class_name, n_samples=10)
        if score is not None:
            scores.append(score)
            print(f"  {class_name:<55} {score:.4f}")
    if scores:
        mean = np.mean(scores)
        jaccard_results[model_name] = mean
        print(f"  Mean Jaccard Index: {mean:.4f}")

print("\n── Summary (Table VI — left column) ──────────────")
for model_name, score in jaccard_results.items():
    print(f"  {model_name:<15} {score:.4f}")

## 7. Explainability Score per model

Reproduces Table VI (right column) of the paper.

In [ ]:
def compute_explainability_scores(
    model,
    model_name: str,
    n_samples_per_class: int = 10,
    threshold: float = 0.5,
) -> dict:
    """
    Compute Explainability Score for every disease class.
    Does NOT require annotation masks — only images.
    """
    scores = {}
    for class_name in CLASS_NAMES:
        class_dir = os.path.join(VALID_DIR, class_name)
        if not os.path.exists(class_dir):
            continue

        img_files = [f for f in os.listdir(class_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        img_files = img_files[:n_samples_per_class]

        heatmaps = []
        for img_file in img_files:
            img_path  = os.path.join(class_dir, img_file)
            img_array = load_image(img_path)
            heatmap   = grad_cam(model, img_array, LAST_CONV_LAYERS[model_name])
            heatmaps.append(heatmap)

        if len(heatmaps) >= 2:
            scores[class_name] = explainability_score(heatmaps, threshold)

    return scores


print("Explainability Score function defined.")

In [ ]:
# Compute Explainability Score for all four models
# Expected output matches Table VI of the paper:
# Xception: 0.39, DenseNet121: 0.34, MobileNetV2: 0.29, InceptionV3: 0.25

es_results = {}
for model_name, model in models.items():
    print(f"\n── {model_name} ─────────────────────")
    scores = compute_explainability_scores(model, model_name, n_samples_per_class=10)
    for class_name, score in scores.items():
        print(f"  {class_name:<55} {score:.4f}")
    mean_score = np.mean(list(scores.values()))
    es_results[model_name] = mean_score
    print(f"  Mean Explainability Score: {mean_score:.4f}")

print("\n── Summary (Table VI — right column) ─────────────")
for model_name, score in es_results.items():
    print(f"  {model_name:<15} {score:.4f}")

## 8. Annotation pipeline — generate LabelMe sampling list

This cell generates `annotation_list.csv` — the list of images we selected  
for manual annotation in LabelMe (25 per class = 250 total annotations).

In [ ]:
SAMPLES_PER_CLASS = 25
OUTPUT_CSV        = "annotation_list.csv"

random.seed(42)  # Reproducible sampling

with open(OUTPUT_CSV, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Class", "ImageName"])

    for class_name in CLASS_NAMES:
        class_dir = os.path.join(VALID_DIR, class_name)
        images    = [f for f in os.listdir(class_dir) if f.lower().endswith((".jpg", ".jpeg"))]
        selected  = random.sample(images, min(SAMPLES_PER_CLASS, len(images)))
        for img in selected:
            writer.writerow([class_name, img])
        print(f"  {class_name}: {len(selected)} images selected")

print(f"\nSaved: {OUTPUT_CSV}")
print("Next step: open each image in LabelMe and annotate the lesion region.")
print("LabelMe install: pip install labelme")

## 9. Organise LabelMe masks into class folders

Run this after completing LabelMe annotations.  
Moves masks from LabelMe's flat output into `organized_masks/ClassName/` structure.

In [ ]:
import shutil
import pandas as pd

RAW_MASKS_DIR  = os.path.join(DATA_DIR, "masks")     # LabelMe raw output
ORG_MASKS_DIR  = os.path.join(DATA_DIR, "organized_masks")

annotation_df = pd.read_csv("annotation_list.csv")

# Create class directories
for class_name in annotation_df["Class"].unique():
    os.makedirs(os.path.join(ORG_MASKS_DIR, class_name), exist_ok=True)

moved = 0
for mask_folder in os.listdir(RAW_MASKS_DIR):
    folder_path = os.path.join(RAW_MASKS_DIR, mask_folder)
    if not os.path.isdir(folder_path):
        continue

    # Match mask folder to class via annotation CSV
    base_name = mask_folder.split("___")[0]
    match = annotation_df[annotation_df["ImageName"].str.contains(base_name, regex=False)]
    if match.empty:
        continue

    class_name = match.iloc[0]["Class"]
    for file_name in ["label.png", "label_viz.png", "img.png"]:
        src = os.path.join(folder_path, file_name)
        dst = os.path.join(ORG_MASKS_DIR, class_name, f"{base_name}_{file_name}")
        if os.path.exists(src):
            shutil.copy2(src, dst)
            if file_name == "label.png":
                # Rename label.png to _mask.png for easy lookup
                mask_dst = os.path.join(ORG_MASKS_DIR, class_name, f"{base_name}_mask.png")
                shutil.copy2(src, mask_dst)
            moved += 1

print(f"Organised {moved} files into {ORG_MASKS_DIR}")